# 🚀 Day 5: Noise & Reality (NISQ Era)

## 14-Day Quantum DevRel Bootcamp

**Goal:** Understand why real quantum computers make errors and how noise limits everything we do in the NISQ era.

**Today's Deliverables:**
1. ✅ Density matrices — pure vs mixed states
2. ✅ Kraus operator noise channels (bit-flip, phase-flip, depolarizing, amplitude damping)
3. ✅ Visualize noise effects on the Bloch sphere
4. ✅ Qiskit Aer noise models
5. ✅ Ideal vs noisy simulation comparison
6. ✅ Understand T1/T2 coherence times and fidelity scaling

**Key insight:** Every gate adds noise. A circuit with depth $d$ and error rate $\varepsilon$ has fidelity $\sim (1-\varepsilon)^d$ — exponential decay. This is why NISQ is hard.

---

## Section 1: Density Matrices — The Language of Noise

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix, Operator, state_fidelity
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error

print("✅ All imports loaded!")
print()

# Standard matrices
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
PAULIS = [X, Y, Z]

# State vectors
STATES = {
    '|0⟩': np.array([1, 0], dtype=complex),
    '|1⟩': np.array([0, 1], dtype=complex),
    '|+⟩': np.array([1, 1], dtype=complex) / np.sqrt(2),
    '|−⟩': np.array([1, -1], dtype=complex) / np.sqrt(2),
    '|+i⟩': np.array([1, 1j], dtype=complex) / np.sqrt(2),
}

def dm(psi):
    """Build density matrix |ψ⟩⟨ψ| from state vector."""
    return np.outer(psi, psi.conj())

def purity(rho):
    """Tr(ρ²)"""
    return float(np.real(np.trace(rho @ rho)))

def bloch(rho):
    """Extract Bloch vector (x, y, z) from density matrix."""
    return tuple(float(np.real(np.trace(rho @ P))) for P in PAULIS)

print("📐 Density Matrices: Pure vs Mixed States")
print("=" * 55)
print()

# Pure states
print("Pure states (ρ = |ψ⟩⟨ψ|, Tr(ρ²) = 1):")
for name, psi in STATES.items():
    rho = dm(psi)
    b = bloch(rho)
    print(f"  {name}: Bloch = ({b[0]:+.2f}, {b[1]:+.2f}, {b[2]:+.2f}), purity = {purity(rho):.4f}")

print()

# Mixed states
print("Mixed states (classical mixtures, Tr(ρ²) < 1):")
mixtures = {
    '50/50 |0⟩,|1⟩': 0.5 * dm(STATES['|0⟩']) + 0.5 * dm(STATES['|1⟩']),
    '50/50 |+⟩,|−⟩': 0.5 * dm(STATES['|+⟩']) + 0.5 * dm(STATES['|−⟩']),
    '70/30 |0⟩,|1⟩': 0.7 * dm(STATES['|0⟩']) + 0.3 * dm(STATES['|1⟩']),
}
for name, rho in mixtures.items():
    b = bloch(rho)
    print(f"  {name}: Bloch = ({b[0]:+.2f}, {b[1]:+.2f}, {b[2]:+.2f}), purity = {purity(rho):.4f}")

print()
print("💡 Key insight: 50/50 |0⟩,|1⟩ and 50/50 |+⟩,|−⟩ give the")
print("   SAME density matrix (I/2). Mixed states erase the 'recipe'.")

---
## Section 2: Quantum Noise Channels

Noise is mathematically described by **quantum channels** using **Kraus operators**:

$$\mathcal{E}(\rho) = \sum_i K_i \rho K_i^\dagger \qquad \text{where } \sum_i K_i^\dagger K_i = I$$

### The Four Standard Channels

| Channel | Physical Model | Effect on Bloch Sphere |
| :--- | :--- | :--- |
| Bit flip | Random X errors | Shrinks Z axis |
| Phase flip | Random Z errors (T2) | Shrinks X-Y plane |
| Depolarizing | Random Pauli errors | Uniform shrinkage |
| Amplitude damping | T1 energy relaxation | Shrinks + drifts to $\vert 0\rangle$ |

In [ ]:
print("🔊 Quantum Noise Channels")
print("=" * 55)
print()

def apply_channel(rho, kraus_ops):
    """ε(ρ) = Σ Kᵢ ρ Kᵢ†"""
    return sum(K @ rho @ K.conj().T for K in kraus_ops)

def bit_flip(p):
    return [np.sqrt(1-p) * I2, np.sqrt(p) * X]

def phase_flip(p):
    return [np.sqrt(1-p) * I2, np.sqrt(p) * Z]

def depolarizing(p):
    return [np.sqrt(1 - 3*p/4) * I2, np.sqrt(p/4) * X,
            np.sqrt(p/4) * Y, np.sqrt(p/4) * Z]

def amplitude_damping(gamma):
    K0 = np.array([[1, 0], [0, np.sqrt(1-gamma)]], dtype=complex)
    K1 = np.array([[0, np.sqrt(gamma)], [0, 0]], dtype=complex)
    return [K0, K1]

def phase_damping(lam):
    K0 = np.array([[1, 0], [0, np.sqrt(1-lam)]], dtype=complex)
    K1 = np.array([[0, 0], [0, np.sqrt(lam)]], dtype=complex)
    return [K0, K1]

# Verify completeness for each channel
print("Channel completeness (Σ K†K = I):")
channels = {
    'Bit flip (p=0.1)': bit_flip(0.1),
    'Phase flip (p=0.1)': phase_flip(0.1),
    'Depolarizing (p=0.1)': depolarizing(0.1),
    'Amplitude damping (γ=0.1)': amplitude_damping(0.1),
    'Phase damping (λ=0.1)': phase_damping(0.1),
}
for name, kraus in channels.items():
    completeness = sum(K.conj().T @ K for K in kraus)
    ok = np.allclose(completeness, I2)
    print(f"  {'✅' if ok else '❌'} {name}")

print()

# Show effects on |+⟩
rho_plus = dm(STATES['|+⟩'])
print("Effect of each channel on |+⟩ (p=0.3):")
print(f"  {'Channel':<25s} {'Bloch (x,y,z)':<25s} {'Purity':>8s}")
print("-" * 60)
print(f"  {'Original |+⟩':<25s} {str(bloch(rho_plus)):<25s} {purity(rho_plus):8.4f}")
for name, channel_fn in [('Bit flip', bit_flip), ('Phase flip', phase_flip),
                          ('Depolarizing', depolarizing),
                          ('Amplitude damping', amplitude_damping),
                          ('Phase damping', phase_damping)]:
    noisy = apply_channel(rho_plus, channel_fn(0.3))
    b = bloch(noisy)
    b_str = f"({b[0]:+.3f}, {b[1]:+.3f}, {b[2]:+.3f})"
    print(f"  {name:<25s} {b_str:<25s} {purity(noisy):8.4f}")

---
## Section 3: Visualizing Noise on the Bloch Sphere

Each noise channel deforms the Bloch sphere differently:
- **Depolarizing**: uniform shrinkage → sphere becomes a smaller sphere
- **Amplitude damping**: shrinkage + drift toward $\vert 0\rangle$ → sphere becomes an egg shape pulled toward the north pole
- **Phase damping**: shrinks X-Y plane → sphere becomes a vertical line
- **Bit flip**: shrinks Z axis → sphere becomes a horizontal disk

In [ ]:
# ── Bloch sphere helpers ──
def make_sphere_wireframe():
    """Translucent Bloch sphere."""
    u = np.linspace(0, 2*np.pi, 30)
    v = np.linspace(0, np.pi, 30)
    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones(len(u)), np.cos(v))
    traces = [go.Surface(
        x=xs, y=ys, z=zs, opacity=0.07,
        colorscale=[[0, 'lightgray'], [1, 'lightgray']],
        showscale=False, hoverinfo='skip'
    )]
    for coords, color, label in [
        ([1.3,0,0], 'red', 'X'), ([0,1.3,0], 'green', 'Y'), ([0,0,1.3], 'blue', 'Z|0⟩')]:
        traces.append(go.Scatter3d(
            x=[0, coords[0]], y=[0, coords[1]], z=[0, coords[2]],
            mode='lines+text', text=['', label], textposition='top center',
            line=dict(color=color, width=2), showlegend=False, hoverinfo='skip'))
    return traces

print("✅ Bloch sphere helpers defined.")

In [ ]:
# ── Track Bloch vector under increasing noise ──
print("👁️ Noise Trajectories on the Bloch Sphere")
print("=" * 55)

# Sample many initial pure states on the Bloch sphere
def sample_bloch_states(n=50):
    """Generate n random pure states uniformly on Bloch sphere."""
    states = []
    for _ in range(n):
        theta = np.arccos(1 - 2*np.random.random())
        phi = 2 * np.pi * np.random.random()
        psi = np.array([np.cos(theta/2), np.exp(1j*phi)*np.sin(theta/2)])
        states.append(psi)
    return states

np.random.seed(42)
initial_states = sample_bloch_states(80)

noise_channels = {
    'Depolarizing': depolarizing,
    'Amplitude Damping': amplitude_damping,
    'Phase Damping': phase_damping,
    'Bit Flip': bit_flip,
}

noise_strength = 0.6  # Moderate noise to see the effect

fig = make_subplots(
    rows=1, cols=4,
    specs=[[{'type': 'scene'}]*4],
    subplot_titles=[f'{name} (p={noise_strength})' for name in noise_channels],
)

for col, (name, channel_fn) in enumerate(noise_channels.items(), 1):
    # Add wireframe sphere
    u = np.linspace(0, 2*np.pi, 25)
    v = np.linspace(0, np.pi, 25)
    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones(len(u)), np.cos(v))
    fig.add_trace(go.Surface(
        x=xs, y=ys, z=zs, opacity=0.05,
        colorscale=[[0, 'lightgray'], [1, 'lightgray']],
        showscale=False, hoverinfo='skip'
    ), row=1, col=col)

    # Original states (faint)
    orig_xyz = [bloch(dm(psi)) for psi in initial_states]
    fig.add_trace(go.Scatter3d(
        x=[p[0] for p in orig_xyz], y=[p[1] for p in orig_xyz],
        z=[p[2] for p in orig_xyz],
        mode='markers', marker=dict(size=2, color='lightblue', opacity=0.3),
        name='Original', showlegend=(col==1)
    ), row=1, col=col)

    # Noisy states
    noisy_xyz = [bloch(apply_channel(dm(psi), channel_fn(noise_strength)))
                 for psi in initial_states]
    fig.add_trace(go.Scatter3d(
        x=[p[0] for p in noisy_xyz], y=[p[1] for p in noisy_xyz],
        z=[p[2] for p in noisy_xyz],
        mode='markers', marker=dict(size=3, color='red', opacity=0.7),
        name='After noise', showlegend=(col==1)
    ), row=1, col=col)

scene = dict(
    xaxis=dict(range=[-1.3, 1.3], title='X'),
    yaxis=dict(range=[-1.3, 1.3], title='Y'),
    zaxis=dict(range=[-1.3, 1.3], title='Z'),
    aspectmode='cube',
)
fig.update_layout(
    title='How Different Noise Channels Deform the Bloch Sphere',
    scene=scene, scene2=scene, scene3=scene, scene4=scene,
    width=1600, height=450,
)
fig.show()

print()
print("Blue dots: original pure states (on the sphere surface)")
print("Red dots: after noise (inside the sphere = mixed states)")
print()
print("💡 Depolarizing: uniform shrinkage toward center")
print("   Amp damping: shrinks AND drifts toward |0⟩ (north pole)")
print("   Phase damping: X-Y plane collapses (lose coherence)")
print("   Bit flip: Z axis collapses (lose population info)")

---
## Section 4: T1 and T2 Coherence Times

Real qubit noise is dominated by two timescales:

| Parameter | Name | Physical Process | Channel Model |
| :--- | :--- | :--- | :--- |
| $T_1$ | Energy relaxation | $\vert 1\rangle \to \vert 0\rangle$ spontaneous decay | Amplitude damping |
| $T_2$ | Dephasing | Phase coherence loss | Phase damping |

**Constraint:** $T_2 \leq 2 \cdot T_1$ (always)

**Typical values (IBM superconducting):**
- $T_1 \approx 100{-}300\,\mu s$
- $T_2 \approx 50{-}200\,\mu s$
- Single-qubit gate time: $\sim 35\,ns$
- CNOT gate time: $\sim 300{-}600\,ns$

This means you get roughly $T_2 / t_{\text{gate}} \approx 200\mu s / 0.3\mu s \approx 660$ CNOTs before coherence is lost.

In [ ]:
print("⏱️ T1/T2 Coherence Decay")
print("=" * 55)
print()

# Simulate T1 decay: |1⟩ over time
T1 = 200e-6  # 200 μs
times = np.linspace(0, 5*T1, 100)
gammas = 1 - np.exp(-times / T1)

p1_t1 = []  # probability of |1⟩ over time
for gamma in gammas:
    rho = apply_channel(dm(STATES['|1⟩']), amplitude_damping(gamma))
    p1_t1.append(rho[1, 1].real)

# Simulate T2 dephasing: |+⟩ coherence over time
T2 = 100e-6  # 100 μs
lambdas = 1 - np.exp(-times / T2)

coherence_t2 = []  # off-diagonal magnitude
for lam in lambdas:
    rho = apply_channel(dm(STATES['|+⟩']), phase_damping(min(lam, 1.0)))
    coherence_t2.append(abs(rho[0, 1]))

# Plot
fig = make_subplots(rows=1, cols=2,
    subplot_titles=('T1 Decay: P(|1⟩) vs Time', 'T2 Dephasing: Coherence vs Time'))

fig.add_trace(go.Scatter(
    x=times*1e6, y=p1_t1, mode='lines',
    name='P(|1⟩)', line=dict(color='#e74c3c', width=3)
), row=1, col=1)
fig.add_vline(x=T1*1e6, line_dash='dash', line_color='gray',
              annotation_text=f'T1={T1*1e6:.0f}μs', row=1, col=1)

fig.add_trace(go.Scatter(
    x=times*1e6, y=coherence_t2, mode='lines',
    name='|ρ₀₁|', line=dict(color='#3498db', width=3)
), row=1, col=2)
fig.add_vline(x=T2*1e6, line_dash='dash', line_color='gray',
              annotation_text=f'T2={T2*1e6:.0f}μs', row=1, col=2)

fig.update_xaxes(title_text='Time (μs)')
fig.update_layout(width=1000, height=400, showlegend=True)
fig.show()

print()
print(f"T1 = {T1*1e6:.0f} μs: |1⟩ population decays exponentially to |0⟩")
print(f"T2 = {T2*1e6:.0f} μs: off-diagonal coherence decays to zero")
print(f"Gate budget: ~{T2/300e-9:.0f} CNOTs before coherence lost")
print()
print("💡 This is the NISQ constraint: you have a limited 'coherence budget'")
print("   to complete your computation before noise overwhelms the signal.")

---
## Section 5: Qiskit Aer Noise Models

Qiskit Aer lets you simulate circuits with realistic noise models:
- **Depolarizing errors** on each gate
- **Thermal relaxation** (T1/T2) during gate execution
- **Measurement errors** (readout bit flips)

This bridges the gap between ideal simulation and real hardware.

In [ ]:
print("🔬 Qiskit Aer Noise Models")
print("=" * 55)
print()

# Build a noise model with depolarizing errors
def make_noise_model(p1=0.001, p2=0.01):
    model = NoiseModel()
    model.add_all_qubit_quantum_error(
        depolarizing_error(p1, 1), ['h', 'x', 'sx', 'rz', 'ry', 's', 't', 'u'])
    model.add_all_qubit_quantum_error(
        depolarizing_error(p2, 2), ['cx', 'cz'])
    return model

# ── Compare ideal vs noisy Bell state ──
qc_bell = QuantumCircuit(2, 2)
qc_bell.h(0)
qc_bell.cx(0, 1)
qc_bell.measure([0, 1], [0, 1])

print("Bell state circuit:")
print(qc_bell.draw())
print()

shots = 8192

# Ideal
sim_ideal = AerSimulator()
ideal_counts = sim_ideal.run(qc_bell, shots=shots).result().get_counts()

# Noisy (mild)
mild_noise = make_noise_model(0.001, 0.01)
noisy_mild = AerSimulator(noise_model=mild_noise).run(qc_bell, shots=shots).result().get_counts()

# Noisy (severe)
severe_noise = make_noise_model(0.01, 0.1)
noisy_severe = AerSimulator(noise_model=severe_noise).run(qc_bell, shots=shots).result().get_counts()

print(f"{'Outcome':<10s} {'Ideal':>8s} {'Mild Noise':>12s} {'Severe Noise':>14s}")
print("-" * 50)
for outcome in ['00', '01', '10', '11']:
    i = ideal_counts.get(outcome, 0)
    m = noisy_mild.get(outcome, 0)
    s = noisy_severe.get(outcome, 0)
    print(f"|{outcome}⟩      {i:>8d} {m:>12d} {s:>14d}")

print()
# Fidelity proxy
for label, counts in [('Ideal', ideal_counts), ('Mild', noisy_mild), ('Severe', noisy_severe)]:
    correct = counts.get('00', 0) + counts.get('11', 0)
    fid = correct / shots
    print(f"  {label}: correct outcomes = {fid*100:.1f}%")

print()
print("💡 With 1% CNOT error: mild degradation.")
print("   With 10% CNOT error: significant noise contamination.")

In [ ]:
# ── Fidelity vs noise level ──
print("📉 Fidelity vs CNOT Error Rate")
print("=" * 55)
print()

error_rates = np.linspace(0, 0.3, 20)
fidelities = []

for p2 in error_rates:
    noise = make_noise_model(0.001, p2)
    sim = AerSimulator(noise_model=noise)
    counts = sim.run(qc_bell, shots=4096).result().get_counts()
    correct = counts.get('00', 0) + counts.get('11', 0)
    fidelities.append(correct / 4096)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=error_rates * 100, y=fidelities,
    mode='lines+markers', name='Measured fidelity',
    line=dict(color='#e74c3c', width=3),
    marker=dict(size=6)
))

# Theoretical: fidelity ≈ (1-p)^n_gates for small p
theoretical = [(1-p)**1 for p in error_rates]  # 1 CNOT
fig.add_trace(go.Scatter(
    x=error_rates * 100, y=theoretical,
    mode='lines', name='Theoretical (1-p)¹',
    line=dict(color='gray', width=2, dash='dash')
))

fig.add_hline(y=0.5, line_dash='dot', line_color='orange',
              annotation_text='Random guessing')

fig.update_layout(
    title='Bell State Fidelity vs CNOT Error Rate',
    xaxis_title='CNOT Error Rate (%)',
    yaxis_title='Fidelity (P(correct))',
    width=800, height=500,
    yaxis=dict(range=[0.3, 1.05]),
)
fig.show()

print("💡 Even one noisy CNOT degrades fidelity significantly.")
print("   For deeper circuits the decay is exponential: F ∝ (1-ε)^d")

---
## Section 6: Circuit Depth and the NISQ Wall

For a circuit with $d$ noisy layers (each with error rate $\varepsilon$):

$$F \approx (1 - \varepsilon)^d \approx e^{-\varepsilon d}$$

The "useful circuit depth" is roughly $d_{\max} \approx 1/\varepsilon$:

| CNOT Error Rate | Max Useful Depth | What You Can Run |
| :--- | :--- | :--- |
| 1% | ~100 layers | Simple VQE, QAOA p=1 |
| 0.1% | ~1000 layers | Deeper variational circuits |
| 0.01% | ~10,000 layers | Getting close to error correction threshold |

This is the **NISQ wall** — the hard limit on what current hardware can do.

In [ ]:
print("🧱 The NISQ Wall: Fidelity vs Circuit Depth")
print("=" * 55)
print()

# Simulate GHZ circuits of increasing depth
depths = range(1, 20)
error_scenarios = {
    '0.1% CNOT error': 0.001,
    '1% CNOT error': 0.01,
    '5% CNOT error': 0.05,
}

fig = go.Figure()
colors = ['#2ecc71', '#f39c12', '#e74c3c']

for (label, p2), color in zip(error_scenarios.items(), colors):
    fids = []
    noise = make_noise_model(0.001, p2)
    sim = AerSimulator(noise_model=noise)

    for n in depths:
        # GHZ-n circuit has n-1 CNOTs → depth scales linearly
        qc = QuantumCircuit(n, n)
        qc.h(0)
        for i in range(n - 1):
            qc.cx(i, i + 1)
        qc.measure(range(n), range(n))

        counts = sim.run(qc, shots=4096).result().get_counts()
        all_0 = '0' * n
        all_1 = '1' * n
        correct = counts.get(all_0, 0) + counts.get(all_1, 0)
        fids.append(correct / 4096)

    fig.add_trace(go.Scatter(
        x=list(depths), y=fids, mode='lines+markers',
        name=label, line=dict(color=color, width=3),
        marker=dict(size=6)
    ))

fig.add_hline(y=0.5, line_dash='dot', line_color='gray',
              annotation_text='Random (useless)')

fig.update_layout(
    title='GHZ State Fidelity vs Number of Qubits (= Circuit Depth)',
    xaxis_title='Number of Qubits (n)',
    yaxis_title='Fidelity (P(correct))',
    width=900, height=500,
    yaxis=dict(range=[0, 1.05]),
)
fig.show()

print("💡 At 5% error, even a 5-qubit GHZ state has <50% fidelity.")
print("   At 0.1% error, you can reach ~20 qubits before degradation.")
print("   This is why error rates are THE metric for quantum hardware.")

---
## 🎯 Day 5 Summary

### What you learned today:
1. **Density Matrices** — The language of open quantum systems: $\rho = \sum p_i |\psi_i\rangle\langle\psi_i|$
2. **Purity** — $\text{Tr}(\rho^2) = 1$ (pure) vs $1/d$ (maximally mixed)
3. **Kraus Operators** — $\mathcal{E}(\rho) = \sum K_i \rho K_i^\dagger$ with completeness $\sum K_i^\dagger K_i = I$
4. **Noise Channels** — Bit flip, phase flip, depolarizing, amplitude damping, phase damping
5. **Bloch Sphere Deformation** — Each channel shrinks the sphere differently
6. **T1/T2 Times** — Energy relaxation and dephasing timescales
7. **Aer Noise Models** — Simulating circuits with realistic gate errors
8. **NISQ Wall** — Fidelity $\sim (1-\varepsilon)^d$ limits useful circuit depth

### Key takeaway for interviews:
> "Real qubits are always in mixed states due to noise. The fidelity of a circuit with depth $d$ and error rate $\varepsilon$ decays as $(1-\varepsilon)^d$ — exponentially. With current CNOT error rates around 0.1-1%, you get at most a few hundred useful gate layers. This is why the NISQ era focuses on shallow variational circuits (VQE, QAOA) and error mitigation techniques. The path forward is either reducing error rates below the fault-tolerant threshold (~0.1%) or developing better error mitigation."

---
**Tomorrow (Day 6):** Grover's Search Algorithm — your first real quantum algorithm with provable quadratic speedup.